# 03 - Sampling Techniques

本 notebook 探索扩散模型的采样技巧和可视化。

**前置条件**: 先运行前两个 notebook 训练 VAE 和扩散模型。

In [ ]:
import sys
sys.path.append('..')

import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from models.vae import VAE
from models.unet import UNet
from models.diffusion import GaussianDiffusion

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. 加载模型

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

vae = VAE(latent_dim=1024).to(device)
vae.load_state_dict(torch.load('../checkpoints/vae.pt', map_location=device))
vae.eval()

unet = UNet(in_channels=64, base_ch=128).to(device)
unet.load_state_dict(torch.load('../checkpoints/unet.pt', map_location=device))

diffusion = GaussianDiffusion(unet, timesteps=1000).to(device)
print('Models loaded successfully')

## 2. 可视化去噪过程

观察模型如何从纯噪声逐步生成数字。

In [ ]:
diffusion.eval()
with torch.no_grad():
    # 生成一个样本，保存中间步骤
    z, intermediates = diffusion.sample_with_progress(
        (1, 64, 4, 4), device, every_n=100
    )
    
    # 解码所有中间步骤
    decoded_steps = []
    for inter in intermediates:
        inter = inter.to(device)
        inter = inter.view(-1, 1024)  # reshape for VAE decoder
        img = vae.decode(inter)
        decoded_steps.append(img.cpu())

# 绘制去噪过程
n_steps = len(decoded_steps)
fig, axes = plt.subplots(1, n_steps, figsize=(n_steps * 2, 2))
for i, ax in enumerate(axes):
    ax.imshow(decoded_steps[i][0, 0] * 0.5 + 0.5, cmap='gray')
    ax.set_title(f'Step {i*100}', fontsize=10)
    ax.axis('off')
plt.suptitle('Denoising Process: Noise → Digit', fontsize=14)
plt.tight_layout()
plt.show()

## 3. 生成多样本网格

In [ ]:
with torch.no_grad():
    z = diffusion.sample((32, 64, 4, 4), device)
    z = z.view(-1, 1024)  # reshape for VAE decoder
    generated = vae.decode(z)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i, 0].cpu() * 0.5 + 0.5, cmap='gray')
    ax.axis('off')
plt.suptitle('Generated Samples Grid', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Latent Space 插值采样

在两个随机噪声之间插值，观察生成结果的渐变。

In [ ]:
with torch.no_grad():
    # 编码两个真实图像得到 latent
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    dataset = datasets.MNIST(root='../data', train=True, download=True, transform=transform)
    
    # 找一个 '0' 和一个 '7'
    images, labels = dataset[0], dataset[1]
    x_0 = images[0].unsqueeze(0).unsqueeze(0).to(device) if isinstance(images, tuple) else images.unsqueeze(0).to(device)
    
    # 重新获取
    loader_iter = iter(DataLoader(dataset, batch_size=64, shuffle=True))
    imgs, lbls = next(loader_iter)
    idx_a = (lbls == 0).nonzero(as_tuple=True)[0][0]
    idx_b = (lbls == 7).nonzero(as_tuple=True)[0][0]
    
    z_a, _, _ = vae.encode(imgs[idx_a:idx_a+1].to(device))
    z_b, _, _ = vae.encode(imgs[idx_b:idx_b+1].to(device))
    
    # 线性插值
    n_steps = 10
    alphas = torch.linspace(0, 1, n_steps).to(device)
    z_interp = torch.cat([z_a * (1-a) + z_b * a for a in alphas])
    
    # 解码 (不需要扩散模型，直接插值 latent space)
    decoded = vae.decode(z_interp)

fig, axes = plt.subplots(1, n_steps, figsize=(20, 2))
for i, ax in enumerate(axes):
    ax.imshow(decoded[i, 0].cpu() * 0.5 + 0.5, cmap='gray')
    ax.set_title(f'a={alphas[i]:.1f}')
    ax.axis('off')
plt.suptitle('Latent Space Interpolation (VAE only)', fontsize=14)
plt.show()

## 5. 温度缩放

通过缩放初始噪声的方差来控制生成的多样性。

In [ ]:
temperatures = [0.5, 0.75, 1.0, 1.25, 1.5]

fig, axes = plt.subplots(len(temperatures), 8, figsize=(16, len(temperatures) * 2))

for row, temp in enumerate(temperatures):
    with torch.no_grad():
        # 温度缩放初始噪声 (不是缩放输出)
        z = diffusion.sample((8, 64, 4, 4), device, temperature=temp)
        z = z.view(-1, 1024)  # reshape for VAE decoder
        generated = vae.decode(z)
    
    for col, ax in enumerate(axes[row]):
        ax.imshow(generated[col, 0].cpu() * 0.5 + 0.5, cmap='gray')
        ax.axis('off')
    axes[row, 0].set_ylabel(f'T={temp}', fontsize=12, rotation=0, labelpad=40)

plt.suptitle('Temperature Scaling Effect', fontsize=14)
plt.tight_layout()
plt.show()

## 6. 创建去噪过程 GIF

In [ ]:
from utils.visualization import create_sampling_gif

with torch.no_grad():
    _, intermediates = diffusion.sample_with_progress(
        (1, 64, 4, 4), device, every_n=50
    )
    
    # 解码
    decoded_gif = []
    for inter in intermediates:
        inter = inter.to(device)
        inter = inter.view(-1, 1024)  # reshape for VAE decoder
        img = vae.decode(inter)
        decoded_gif.append(img)

create_sampling_gif(decoded_gif, '../outputs/denoising_process.gif', duration=150)
print('GIF saved to ../outputs/denoising_process.gif')

## 7. 比较不同时间步的去噪效果

观察从不同时间步开始去噪的效果差异。

In [ ]:
start_steps = [100, 300, 500, 700, 900]

fig, axes = plt.subplots(1, len(start_steps), figsize=(15, 3))

for i, start_t in enumerate(start_steps):
    with torch.no_grad():
        # 从噪声开始，但只运行部分去噪步骤
        z = torch.randn(1, 64, 4, 4, device=device)
        for t in reversed(range(start_t)):
            t_batch = torch.tensor([t], device=device, dtype=torch.long)
            z = diffusion.p_sample(z, t_batch)
        
        z = z.view(-1, 1024)  # reshape for VAE decoder
        img = vae.decode(z)
    
    axes[i].imshow(img[0, 0].cpu() * 0.5 + 0.5, cmap='gray')
    axes[i].set_title(f'Start from t={start_t}')
    axes[i].axis('off')

plt.suptitle('Partial Denoising (Starting from Different Timesteps)', fontsize=14)
plt.tight_layout()
plt.show()

## 总结

通过本 notebook，我们探索了：
1. **去噪过程可视化**: 观察模型如何从噪声逐步生成数字
2. **多样本生成**: 批量生成不同的数字
3. **Latent 空间插值**: 在两个噪声之间平滑过渡
4. **温度控制**: 调节生成的多样性
5. **部分去噪**: 观察不同时间步的去噪效果

这些技巧可以帮助你更好地理解和使用扩散模型。